In [54]:
import h5py
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from dataclasses import dataclass

In [66]:
@dataclass
class DESI_DATA:
    #Settings
    tag : str
    ells : tuple
    #Data
    k_vec : np.ndarray
    ell_vec : np.ndarray
    k_theory : dict
    data : np.ndarray
    C : np.ndarray
    Cinv : np.ndarray
    W : np.ndarray

    @classmethod
    def load_desi(cls, tag, ells):
        # 1. File paths
        spectrum_file, cov_file, win_file = load_files(tag)

        # 2. Read everything (all multipoles in the files)
        k, k_edges, data_full       = get_spectrum(spectrum_file)
        cov_edges, C_full           = get_cov(cov_file)
        k_theory, win_edges, W_full = get_win(win_file)

        # 3. Cut to the multipoles in `ells`
        data, C, Cinv, W, data_mask = mask_data(ells, k_edges, data_full,
                                            cov_edges, C_full, win_edges, W_full)

        # 4. Labels for each data entry
        k_vec   = np.concatenate([k[ell] for ell in DATA_ELLS])[data_mask]
        ell_vec = np.concatenate([np.full(len(k[ell]), ell) for ell in DATA_ELLS])[data_mask]

        # 5. Essential checks
        n = len(data)
        assert C.shape == (n, n) and W.shape[0] == n
        assert np.allclose(C, C.T)
        assert np.linalg.eigvalsh(C).min() > 0

        # 6. Package
        return DESI_DATA(tag=tag, ells=ells, k_vec=k_vec, ell_vec=ell_vec,
                                 k_theory=k_theory, data=data, C=C, Cinv=Cinv, W=W)
    

In [56]:
#Settings

tag = "LRG_GCcomb_z0.4-0.6_thetacut0.05" #Depends on choice of Desi Data
ells = ["0","2","4"]                     #Subset of ells used in calculation 0, 2, 4
DATA_ELLS = ["0", "2", "4"]              #Multipole ells in DATA


In [57]:
#Load Data

def load_files(tag):
    DATA = Path("Data")

    spectrum_file = DATA / ("spectrum-poles-rotated-corrected_" + tag + ".h5")
    cov_file = DATA / ("covariance_spectrum-poles-rotated_" + tag + ".h5")
    win_file = DATA / ("window_spectrum-poles-rotated_" + tag + ".h5")

    return spectrum_file, cov_file, win_file



#Check

spectrum_file, cov_file, win_file = load_files(tag)
for f in [spectrum_file, cov_file, win_file]:
    print(f.name, f.exists())

In [58]:
#Get relevant spectrum Data

def get_spectrum(file):
    
    k, k_edges, P = {}, {}, {}

    with h5py.File(file) as h:               # step 1: open the file on disk
        for ell in DATA_ELLS:                              # step 2: Run over all ells
            k[ell] = h[f"{ell}/k"][:]                 # step 3: read the dataset "ell/k", ell = 0,2,4
            k_edges[ell] = h[f"{ell}/k_edges"][:]     # step 4: read the dataset "ell/k_edges", ell = 0,2,4
            P[ell] = h[f"{ell}/value"][:]             # step 5: read the dataset "ell/value", ell = 0,2,4

    data_full = np.concatenate([P[ell] for ell in DATA_ELLS])   # Data should be a vector to calculate r = data - model => chi^2
                                                                # First 36 entries are ell = 0, then ell = 2, then ell = 4
    return k, k_edges, data_full

#Check

k, k_edges, data_full = get_spectrum(spectrum_file)
for ell in DATA_ELLS:
    print(k_edges[ell].shape)
    print(k[ell].shape)

print(data_full.shape) 


In [59]:
#Get relevant covariance Data

def get_cov(file):

    cov_edges = {}


    with h5py.File(file) as h:
        for ell in DATA_ELLS:
            cov_edges[ell] = h[f"observable/{ell}/k_edges"][:] #Get k_edges for the covariance matrix
    
        C_full = h["value"][:]                                 # Get Covariance Matrix

    return cov_edges, C_full
    
#Checks

cov_edges, C_full = get_cov(cov_file)
for ell in DATA_ELLS:
    print(cov_edges[ell].shape)

print(C_full.shape)

In [60]:
#Get relevant window Data

def get_win(file):
    
    win_edges, k_theory = {}, {}


    with h5py.File(file) as h:
        for ell in DATA_ELLS:
            win_edges[ell] = h[f"observable/{ell}/k_edges"][:] 
            k_theory[ell] = h[f"theory/{ell}/k"][:]
        
        W_full = h["value"][:]                                 

    return k_theory, win_edges, W_full
    
#Checks
k_theory, win_edges, W_full = get_win(win_file)

for ell in DATA_ELLS:
    print(win_edges[ell].shape)
    print(k_theory[ell].shape)

print(W_full.shape)

In [61]:
#Mask: Cut data to equal size needed

def make_mask(ells, data_edges, file_edges):                      #file_edges = k_edges, cov_edges, win_edges
    
    mask = []
    
    for ell in DATA_ELLS:
        for e in file_edges[ell]:
            if ell in ells:
                match = np.isclose(e,data_edges[ell]).all(axis=1).any()
            else:
                match = False
            mask.append(match)
            
    return mask

In [68]:
#Final Data: Only part of Data we need (given by ells from Settings)

def mask_data(ells, k_edges, data_full, cov_edges, C_full, win_edges, W_full):
    
    data_mask = make_mask(ells, k_edges, k_edges)                              
    cov_mask = make_mask(ells, k_edges, cov_edges)
    win_mask = make_mask(ells, k_edges, win_edges)
    
    data      = data_full[data_mask]
    C = C_full[np.ix_(cov_mask,cov_mask)]
    C_inv = np.linalg.inv(C) 
    W = W_full[win_mask,:]

    return data, C, C_inv, W, data_mask

data, C, Cinv, W , data_mask = mask_data(ells, k_edges, data_full, cov_edges, C_full, win_edges, W_full)

print(data.shape, C.shape, Cinv.shape, W.shape)

In [63]:
#Check the output

n=len(data)

# Covariance is symmetric and positive definite
print("symmetric:", np.allclose(C, C.T))
print("smallest eigenvalue:", np.linalg.eigvalsh(C).min())      # must be > 0

# Inverse works
Cinv = np.linalg.inv(C)
print("C @ Cinv ≈ identity:", np.allclose(C @ Cinv, np.eye(n)))

# The masks selected the same bins everywhere
data_mask = make_mask(ells, k_edges, k_edges)                              
cov_mask = make_mask(ells, k_edges, cov_edges)
win_mask = make_mask(ells, k_edges, win_edges)
data_edges_all = np.concatenate([k_edges[ell]   for ell in DATA_ELLS])[data_mask]
cov_edges_all  = np.concatenate([cov_edges[ell] for ell in DATA_ELLS])[cov_mask]
win_edges_all  = np.concatenate([win_edges[ell] for ell in DATA_ELLS])[win_mask]
print("cov bins match data:", np.allclose(cov_edges_all, data_edges_all))
print("win bins match data:", np.allclose(win_edges_all, data_edges_all))
print("condition number:", np.linalg.cond(C))

In [64]:
#Plot k*P_ell(k)

k_vec   = np.concatenate([k[ell] for ell in DATA_ELLS])[data_mask]
ell_vec = np.concatenate([np.full(len(k[ell]), ell) for ell in DATA_ELLS])[data_mask]
err     = np.sqrt(np.diag(C))

for ell in ells:
    s = ell_vec == ell                     # True for the entries of this multipole
    plt.errorbar(k_vec[s], k_vec[s] * data[s], yerr=k_vec[s] * err[s],
                 fmt="o", ms=3, label=f"ℓ = {ell}")
plt.axhline(0, c="grey", lw=0.5)
plt.xlabel("k  [h/Mpc]")
plt.ylabel("k · P_ℓ(k)  [(Mpc/h)²]")
plt.legend()

In [73]:
#Check load desi

d0 = DESI_DATA.load_desi(tag, ("0",))
d2 = DESI_DATA.load_desi(tag, ("0","2",))
d4 = DESI_DATA.load_desi(tag, ("0","2","4",))
print(d0.data.shape, d0.C.shape, d0.W.shape)
print(d2.data.shape, d2.C.shape, d2.W.shape)
print(d4.data.shape, d4.C.shape, d4.W.shape)